## Tratamiento de datos faltantes

In [18]:
# Cargamos los paquetes necesarios ACORDARNOS DE HACER EL AMBIENTE VIRTUAL
import matplotlib.pyplot as plt 
import polars as pl
import seaborn as sns
import numpy as np
import pyprojroot
from plotnine import *
import pandas as pd

In [19]:
# Definir la ruta raiz del proyecto
ROOT = pyprojroot.here()

# Cargamos los datos
datos_entrenamiento = pl.read_parquet(ROOT / "datos" / "temporada1.parquet")
datos_validacion = pl.read_parquet(ROOT / "datos" / "temporada2.parquet")

In [20]:
na = (
    datos_entrenamiento
    .null_count()
    .transpose(include_header=True)
    .filter(pl.col("column_0") > 0)
)

na

column,column_0
str,u32
"""pitch_id""",1381
"""release_speed""",367
"""pitch_type""",367
"""pfx_x""",3033
"""pfx_z""",1060
"""plate_x""",367
"""plate_z""",400
"""sz_top""",367
"""sz_bot""",412


In [21]:

datos_con_na = datos_entrenamiento.filter(
    pl.any_horizontal(pl.all().is_null())
)

datos_con_na.height

5131

In [22]:

datos_entrenamiento.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") > 0
).group_by("NA_por_fila").len()

NA_por_fila,len
u32,u32
1,4712
2,52
10,47
9,320


In [23]:
datos_entrenamiento.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") >= 9
)

pitch_id,release_speed,batter,pitcher,description,stand,p_throws,pitch_type,balls,strikes,pfx_x,pfx_z,plate_x,plate_z,sz_top,sz_bot,altura_zona,swing,NA_por_fila
i64,f64,i64,i64,str,str,str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,u32
null,null,5586,5770,"""ball""","""R""","""R""",null,3,0,null,null,null,null,null,null,null,0,10
null,null,6262,5051,"""ball""","""L""","""R""",null,1,0,null,null,null,null,null,null,null,0,10
null,null,5299,5423,"""called_strike""","""R""","""L""",null,1,0,null,null,null,null,null,null,null,0,10
2327986,null,5475,5561,"""ball""","""R""","""L""",null,0,1,null,null,null,null,null,null,null,0,9
2216154,null,5159,5365,"""called_strike""","""R""","""L""",null,0,1,null,null,null,null,null,null,null,0,9
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,null,5938,6330,"""swinging_strike""","""R""","""L""",null,0,0,null,null,null,null,null,null,null,1,10
null,null,6310,6878,"""ball""","""L""","""R""",null,1,1,null,null,null,null,null,null,null,0,10
null,null,6576,6878,"""foul""","""R""","""R""",null,3,2,null,null,null,null,null,null,null,1,10



Se realizó un análisis de los valores faltantes presentes en el conjunto de entrenamiento. Se identificaron 5.131 observaciones con al menos un valor ausente sobre un total de 709.852 registros, lo que representa aproximadamente un 0,72% de los datos. Los faltantes se concentran principalmente en variables relacionadas con las características físicas y espaciales de los lanzamientos, tales como la velocidad (`release_speed`), el tipo de lanzamiento (`pitch_type`), el movimiento de la pelota (`pfx_x` y `pfx_z`) y la ubicación del lanzamiento respecto de la zona de strike (`plate_x`, `plate_z`, `sz_top`, `sz_bot` y `altura_zona`).

Si bien la proporción de observaciones incompletas es reducida, la consigna del trabajo requiere generar una predicción para cada lanzamiento presente en `temporada2.parquet`. En consecuencia, la eliminación de registros con valores faltantes podría ocasionar una pérdida de observaciones durante la etapa de validación y dificultar la generación de predicciones para la totalidad de los lanzamientos.

Por este motivo, se opta por imputar los valores faltantes. Para las variables numéricas se utiliza la mediana calculada sobre los datos de entrenamiento.. En el caso de las variables categóricas, los valores ausentes se reemplazan por la categoría más frecuente observada en el conjunto de entrenamiento, es decir, la moda.

#### Imputación



Luego del análisis realizado, se aplica la estrategia de imputación seleccionada sobre los conjuntos de entrenamiento y validación. Para evitar la incorporación de información proveniente del conjunto de validación, los parámetros de imputación se calculan únicamente utilizando los datos correspondientes a `temporada1.parquet` y posteriormente se aplican sobre ambos conjuntos.

In [24]:
variables_cuantitativas = [
    "release_speed",
    "balls",
    "strikes",
    "pfx_x",
    "pfx_z",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "altura_zona"
]

variables_categoricas = [
    "pitch_type",
    "stand",
    "p_throws"
]

In [25]:
from sklearn.impute import SimpleImputer

# Cuantitativas
imputador_num = SimpleImputer(strategy="median")

datos_entrenamiento[variables_cuantitativas] = imputador_num.fit_transform(
    datos_entrenamiento[variables_cuantitativas]
)

datos_validacion[variables_cuantitativas] = imputador_num.transform(
    datos_validacion[variables_cuantitativas]
)

# Categóricas
# Para que funcione imputador_cat tuve que convertir el dataframe de polars a un dataframe de pandas
datos_entrenamiento = datos_entrenamiento.to_pandas()
datos_validacion = datos_validacion.to_pandas()

imputador_cat = SimpleImputer(strategy="most_frequent")

# Aprende la moda de temporada1 y la aplica
datos_entrenamiento[variables_categoricas] = imputador_cat.fit_transform(
    datos_entrenamiento[variables_categoricas]
)

# Usa las mismas modas aprendidas y las aplica a validación
datos_validacion[variables_categoricas] = imputador_cat.transform(
    datos_validacion[variables_categoricas]
)

Una vez realizada la imputación, se chequea que ya no haya valores nulos presentes en los dataframes de entrenamiento y validación

In [ ]:
datos_entrenamiento[variables_categoricas].isna().sum()

pitch_type    0
stand         0
p_throws      0
dtype: int64

In [27]:
datos_entrenamiento[variables_cuantitativas].isna().sum()

release_speed    0
balls            0
strikes          0
pfx_x            0
pfx_z            0
plate_x          0
plate_z          0
sz_top           0
sz_bot           0
altura_zona      0
dtype: int64

In [29]:
datos_validacion[variables_categoricas].isna().sum()

pitch_type    0
stand         0
p_throws      0
dtype: int64

In [30]:
datos_validacion[variables_cuantitativas].isna().sum()

release_speed    0
balls            0
strikes          0
pfx_x            0
pfx_z            0
plate_x          0
plate_z          0
sz_top           0
sz_bot           0
altura_zona      0
dtype: int64

In [31]:
# Para volver a dataframes de polars
datos_entrenamiento = pl.from_pandas(datos_entrenamiento)
datos_validacion = pl.from_pandas(datos_validacion)

In [32]:
datos_entrenamiento.write_parquet(
    "../datos/temporada1_imputada.parquet"
)

datos_validacion.write_parquet(
    "../datos/temporada2_imputada.parquet"
)